# pyskills.ipynb
> Functions for view/modifying ipynb file notebook cells. Each operation returns unified diffs showing what changed.
> 
> ## Ipynb file cell editing
> 
> Cell tools take an ipynb path and a cell id, e.g:
> 
>     cell_replace_lines('nb.ipynb', cell_id, 2, 3, 'replaced')
>     cell_insert_line('nb.ipynb', cell_id, 0, 'first line')
> 
> Use `view_nb` to view the whole notebook. Use `view_cell` to see a cell's source with line numbers before editing.
> 
> ## Line filtering
> 
> `cell_str_replace`, `cell_strs_replace`, and `cell_del_lines` support `re_filter` and `invert_filter` for targeting only lines matching (or not matching) a regex, like ex's `g//` and `g!//`. Combine with `start_line`/`end_line` to restrict to a region.

In [ ]:
#| default_exp ipynb

In [ ]:
#| export
import difflib,re
from pathlib import Path
from tempfile import TemporaryDirectory

from fastcore.meta import splice_sig
from pyskills.edit import *

In [ ]:
from fastcore.test import test_eq

In [ ]:
#| export
_cell_edit_doc = f"""
Be sure you've called `view_cell(…)` to ensure you know the line nums.

Cell editing standard parameters:

id: Cell id to edit, or list of ids, or 'all' for all messages in file
fname: ipynb to get info for
update_output: If True, replace in output instead of content

returns:
- For single id: diff of changes, or "none: No changes.", or "error: ..."
- For id list (or 'all'): list of tuples of (id,diff) for changed messages"""

In [ ]:
#| export
def _cell_edit(f, name=None):
    def wrapper(fname:str, id:str|list[str], *args, update_output:bool=False, **kw):
        nb = Notebook.open(fname)
        def _one(cid):
            cell = nb[cid]
            text = str(cell.outputs) if update_output else cell.source
            if not text: return f"error: Cell has no {'output' if update_output else 'source'}"
            try: new_text = f(text, *args, **kw)
            except ValueError as e: return f'error: {e}'
            if update_output: cell.outputs = ast.literal_eval(new_text)
            else: nb[cid] = new_text
            diff = '\n'.join(list(difflib.unified_diff(text.splitlines(), new_text.splitlines(), n=1, lineterm=''))[2:])
            return diff or 'none: No changes.'
        if isinstance(id, list) or id == 'all':
            if id == 'all': id = [c.id for c in nb.cells]
            res = [(cid, r) for cid in id if not (r := _one(cid)).startswith(('error:', 'none:'))]
        else: res = _one(id)
        nb.save()
        return res
    res = splice_sig(wrapper, f, 'text')
    if name: res.__name__ = res.__qualname__ = name
    res.__doc__ = (f.__doc__ or '') + _cell_edit_doc
    return res

In [ ]:
#| export
from fastcore.nbio import *

In [ ]:
_tmp = TemporaryDirectory()
_test_content = 'alpha\nbeta\ngamma\ndelta\n'

_nb_path = f'{_tmp.name}/test.ipynb'
_tnb = Notebook(new_nb([_test_content, 'other cell']))
_tnb.save(_nb_path)
_cid, _oid = _tnb[0].id, _tnb[1].id
def _nb_src(): return Notebook.open(_nb_path)[_cid].source
_cid, _oid

('fc4d7e8e', 'b0e7ba69')

In [ ]:
#| export
cell_insert_line = _cell_edit(insert_line, 'cell_insert_line')
cell_str_replace = _cell_edit(str_replace, 'cell_str_replace')
cell_strs_replace = _cell_edit(strs_replace, 'cell_strs_replace')
cell_replace_lines = _cell_edit(replace_lines, 'cell_replace_lines')
cell_del_lines = _cell_edit(del_lines, 'cell_del_lines')

In [ ]:
res = cell_insert_line(_nb_path, _cid, 0, 'first')
test_eq(_nb_src().splitlines()[0], 'first')
print(res)

@@ -1 +1,2 @@


+first


 alpha


In [ ]:
res = cell_str_replace(_nb_path, _cid, 'beta', 'BETA')
assert 'BETA' in _nb_src(), f"Expected 'BETA' in cell"
print(res)

@@ -2,3 +2,3 @@


 alpha


-beta


+BETA


 gamma


In [ ]:
res = cell_strs_replace(_nb_path, _cid, ['gamma', 'delta'], ['GAMMA', 'DELTA'])
test_eq(_nb_src().splitlines()[-2:], ['GAMMA', 'DELTA'])
print(res)

@@ -3,3 +3,3 @@


 BETA


-gamma


-delta


+GAMMA


+DELTA


In [ ]:
res = cell_replace_lines(_nb_path, _cid, 2, 3, 'two\nthree\n')
test_eq(_nb_src().splitlines()[1:3], ['two', 'three'])
print(res)

@@ -1,4 +1,4 @@


 first


-alpha


-BETA


+two


+three


 GAMMA


In [ ]:
res = cell_del_lines(_nb_path, _cid, 1)
test_eq(_nb_src().splitlines()[0], 'two')
print(res)

@@ -1,2 +1 @@


-first


 two


In [ ]:
#| export
def view_cell(fname:str, id:str, nums:bool=True):
    "Show cell source with optional line numbers"
    return Notebook.open(fname).view(id, nums=nums)

`view_cell` displays a cell's source with line numbers, useful for checking content before editing:

In [ ]:
print(view_cell(_nb_path, _cid))

     1 │ two


     2 │ three


     3 │ GAMMA


     4 │ DELTA


In [ ]:
#| export
def view_nb(fname:str, incl_out:bool=False):
    "Show notebook source as concise xml, optionally including output if `incl_out`"
    nb = Notebook.open(fname)
    return repr(nb) if incl_out else nb.concise

In [ ]:
view_nb(_nb_path)

'<nb path="test.ipynb"><code id="fc4d7e8e">two\nthree\nGAMMA\nDELTA</code><code id="b0e7ba69">other cell</code></nb>'

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()